In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, accuracy_score

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Dropout

In [2]:
df = pd.read_csv('final_dataset/multi_label_encoded.csv')

In [3]:
X = df.iloc[:, 0:19].values
y = df.iloc[:, -1].values

In [4]:
# one hot encoding for "Race" and "District"

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [4, 5])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

In [5]:
print(X)

[[0.0 0.0 0.0 ... 284.137 215.2 232.978]
 [0.0 0.0 1.0 ... 284.137 215.2 232.978]
 [0.0 0.0 0.0 ... 284.137 215.2 232.978]
 ...
 [0.0 0.0 1.0 ... 368.791 277.763 300.628]
 [0.0 0.0 1.0 ... 368.791 277.763 300.628]
 [0.0 0.0 1.0 ... 368.791 277.763 300.628]]


In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 1)

In [7]:
# feature scaling

from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [8]:
from sklearn.tree import DecisionTreeClassifier
classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 1)
classifier.fit(X_train, y_train)

DecisionTreeClassifier(criterion='entropy', random_state=1)

In [9]:
y_pred_dt = classifier.predict(X_test)
print(np.concatenate((y_pred_dt.reshape(len(y_pred_dt),1), y_test.reshape(len(y_test),1)),1))

[[1 3]
 [1 3]
 [3 3]
 ...
 [3 1]
 [1 1]
 [3 1]]


In [10]:
# confusion matrix and accuracy for Decision Tree

cm = confusion_matrix(y_test, y_pred_dt)
print(cm)
accuracy_score(y_test, y_pred_dt)

[[   10    93     1   197     4]
 [  125 14256   114 14538   377]
 [    0   145  2278   175     7]
 [  265 15277   182 38400   348]
 [    2   335     3   306    42]]


0.6285550983081847

In [11]:
# 10 fold cross validation for Decision Tree

from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = classifier, X = X_train, y = y_train, cv = 10)
print("Accuracy: {:.2f} %".format(accuracies.mean()*100))
print("Standard Deviation: {:.2f} %".format(accuracies.std()*100))

Accuracy: 62.39 %
Standard Deviation: 0.28 %


In [12]:
# training with random forest classification

from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
classifier.fit(X_train, y_train)

RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)

In [13]:
y_pred_rf = classifier.predict(X_test)
print(np.concatenate((y_pred_rf.reshape(len(y_pred_rf),1), y_test.reshape(len(y_test),1)),1))

[[3 3]
 [3 3]
 [3 3]
 ...
 [1 1]
 [1 1]
 [3 1]]


In [14]:
# confusion matrix for random forest classifier

cm = confusion_matrix(y_test, y_pred_rf)
print(cm)
accuracy_score(y_test, y_pred_rf)

[[    5    86     2   212     0]
 [   18 14880   135 14333    44]
 [    0   430  1720   453     2]
 [   27 12932   120 41372    21]
 [    0   371     3   302    12]]


0.6628829446730681

In [15]:
# 10 fold cross validation for Random Forest

from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = classifier, X = X_train, y = y_train, cv = 10)
print("Accuracy: {:.2f} %".format(accuracies.mean()*100))
print("Standard Deviation: {:.2f} %".format(accuracies.std()*100))

Accuracy: 65.91 %
Standard Deviation: 0.36 %


In [16]:
# training with logistic regression

from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(random_state = 1)
classifier.fit(X_train, y_train)

C:\Users\14104\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(random_state=1)

In [17]:
# predict test result with logistic regression

y_pred_lr = classifier.predict(X_test)
print(np.concatenate((y_pred_lr.reshape(len(y_pred_lr),1), y_test.reshape(len(y_test),1)),1))

[[3 3]
 [3 3]
 [3 3]
 ...
 [3 1]
 [3 1]
 [1 1]]


In [18]:
# confusion matrix for logistic regression

from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred_lr)
print(cm)
accuracy_score(y_test, y_pred_lr)

[[    0    46     0   259     0]
 [    0  9201    56 20153     0]
 [    0  1051    58  1496     0]
 [    0  6322    37 48113     0]
 [    0   265     0   423     0]]


0.6558299039780521

In [19]:
# 10 fold cross validation for Logistic Regression

from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = classifier, X = X_train, y = y_train, cv = 10)
print("Accuracy: {:.2f} %".format(accuracies.mean()*100))
print("Standard Deviation: {:.2f} %".format(accuracies.std()*100))

C:\Users\14104\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\14104\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_

Accuracy: 65.39 %
Standard Deviation: 0.24 %


C:\Users\14104\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [20]:
# training on K-NN

from sklearn.neighbors import KNeighborsClassifier
classifier = KNeighborsClassifier(n_neighbors = 1000, metric = 'minkowski', p = 2)
classifier.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=1000)

In [22]:
y_pred_knn = classifier.predict(X_test)
#print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

In [23]:
# confusion matrix and accuracy score for K-NN

from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred_knn)
print(cm)
accuracy_score(y_test, y_pred_knn)

[[    0    24     0   281     0]
 [    0  5883     0 23527     0]
 [    0   767     0  1838     0]
 [    0  3939     0 50533     0]
 [    0   162     0   526     0]]


0.644901691815272

In [24]:
# 10 fold cross validation for K-NN

from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = classifier, X = X_train, y = y_train, cv = 10)
print("Accuracy: {:.2f} %".format(accuracies.mean()*100))
print("Standard Deviation: {:.2f} %".format(accuracies.std()*100))

Accuracy: 64.20 %
Standard Deviation: 0.27 %


In [25]:
# training on bayesian network

from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train, y_train)

GaussianNB()

In [26]:
y_pred_bn = classifier.predict(X_test)
#print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

In [27]:
# confusion matrix and accuracy on bayesian network

from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred_bn)
print(cm)
accuracy_score(y_test, y_pred_bn)

[[   12     0   253     0    40]
 [  499    72 24279    10  4550]
 [    8     0  2509     0    88]
 [ 2416    70 35777    30 16179]
 [    8     0   505     0   175]]


0.03198445358939186

In [28]:
# 10 fold cross validation for Bayesian Network

from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = classifier, X = X_train, y = y_train, cv = 10)
print("Accuracy: {:.2f} %".format(accuracies.mean()*100))
print("Standard Deviation: {:.2f} %".format(accuracies.std()*100))

Accuracy: 3.04 %
Standard Deviation: 0.29 %


In [29]:
#keras.wrappers is use to implement the k-cross validation
from keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import cross_val_score
from keras.models import Sequential
from keras.layers import Dense

In [30]:
def build_classifier():
	classifier = Sequential()
	
	classifier.add(Dense(
		units = 128,
		activation="relu",
		))
	
	classifier.add(Dense(
		units = 128,
		activation="relu"
		))
	
	classifier.add(Dense(
		units = 1,
		activation="softmax"
		))
	
	classifier.compile(
		optimizer = "adam",
		loss="binary_crossentropy",
		metrics=['accuracy']
		)
	
	return classifier

In [31]:
#this classifier will be use to the 10 different training fold 
#for k-cross validation on 1 test fold
classifier = KerasClassifier(build_fn = build_classifier,
							 batch_size = 32,
							 nb_epoch = 100 )
classifier

C:\Users\14104\AppData\Local\Temp\ipykernel_15528\385065381.py:3: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  classifier = KerasClassifier(build_fn = build_classifier,


In [32]:
accuracies = cross_val_score(
		estimator=classifier,
		X = X_train,
		y = y_train,
		cv=10
		)

638/638 [==============================] - 1s 1ms/step - loss: -32835064.0000 - accuracy: 0.3370


In [33]:
#after we got the accuracies, find the mean
mean = accuracies.mean()
variance = accuracies.std()

In [34]:
# mean from 10 fold cross validation of Neural Network
mean

0.33956006467342376

In [35]:
# std from 10 fold cross validation of Neural Network
variance

0.003057471902448795

In [36]:
# another way to implement ANN
ann = tf.keras.models.Sequential()

In [37]:
# adding first hidden layer
ann.add(tf.keras.layers.Dense(units=128, activation='relu'))

In [38]:
# adding second hidden layer
ann.add(tf.keras.layers.Dense(units=128, activation='relu'))

In [39]:
# adding output layer
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

In [40]:
# compiling the ANN
ann.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [41]:
ann.fit(X_train, y_train, batch_size = 32, epochs = 100)

Epoch 1/100
6379/6379 [==============================] - 9s 1ms/step - loss: -12492755.0000 - accuracy: 0.3394
Epoch 2/100
6379/6379 [==============================] - 9s 1ms/step - loss: -126373936.0000 - accuracy: 0.3396
Epoch 3/100
6379/6379 [==============================] - 10s 2ms/step - loss: -439563680.0000 - accuracy: 0.3396
Epoch 4/100
6379/6379 [==============================] - 10s 2ms/step - loss: -1040621696.0000 - accuracy: 0.3396
Epoch 5/100
6379/6379 [==============================] - 10s 2ms/step - loss: -2015612160.0000 - accuracy: 0.3396
Epoch 6/100
6379/6379 [==============================] - 10s 2ms/step - loss: -3456235008.0000 - accuracy: 0.3396
Epoch 7/100
6379/6379 [==============================] - 10s 2ms/step - loss: -5448130560.0000 - accuracy: 0.3396
Epoch 8/100
6379/6379 [==============================] - 10s 2ms/step - loss: -8092239872.0000 - accuracy: 0.3396
Epoch 9/100
6379/6379 [==============================] - 10s 2ms/step - loss: -11481115648.000

6379/6379 [==============================] - 10s 2ms/step - loss: -5805828997120.0000 - accuracy: 0.3396
Epoch 72/100
6379/6379 [==============================] - 10s 2ms/step - loss: -6058159898624.0000 - accuracy: 0.3396
Epoch 73/100
6379/6379 [==============================] - 10s 2ms/step - loss: -6317800423424.0000 - accuracy: 0.3396
Epoch 74/100
6379/6379 [==============================] - 9s 1ms/step - loss: -6584350539776.0000 - accuracy: 0.3396
Epoch 75/100
6379/6379 [==============================] - 10s 2ms/step - loss: -6858942185472.0000 - accuracy: 0.3396
Epoch 76/100
6379/6379 [==============================] - 10s 2ms/step - loss: -7141292769280.0000 - accuracy: 0.3396
Epoch 77/100
6379/6379 [==============================] - 10s 2ms/step - loss: -7431057833984.0000 - accuracy: 0.3396
Epoch 78/100
6379/6379 [==============================] - 10s 2ms/step - loss: -7728113647616.0000 - accuracy: 0.3396
Epoch 79/100
6379/6379 [==============================] - 10s 2ms/step

In [42]:
# predicting the test results
y_pred_nn = ann.predict(X_test)
y_pred_nn = (y_pred_nn > 0.5)
print(np.concatenate((y_pred_nn.reshape(len(y_pred_nn),1), y_test.reshape(len(y_test),1)),1))

2734/2734 [==============================] - 2s 724us/step
[[1 3]
 [1 3]
 [1 3]
 ...
 [1 1]
 [1 1]
 [1 1]]


In [43]:
# making confusion matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred_nn)
print(cm)
accuracy_score(y_test, y_pred_nn)

[[    0   305     0     0     0]
 [    0 29410     0     0     0]
 [    0  2605     0     0     0]
 [    0 54472     0     0     0]
 [    0   688     0     0     0]]


0.33619112940100593

In [44]:
# computing AUROC and ROC curve values

from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.preprocessing import LabelBinarizer

def multiclass_roc_auc_score(y_test, y_pred, average="macro"):
    lb = LabelBinarizer()
    lb.fit(y_test)
    y_test = lb.transform(y_test)
    y_pred = lb.transform(y_pred)
    return roc_auc_score(y_test, y_pred, average=average)

In [45]:
# AUROC score for Decision Tree
multiclass_roc_auc_score(y_test, y_pred_dt, average="macro")

0.6407551025016491

In [46]:
# AUROC score for random forest
multiclass_roc_auc_score(y_test, y_pred_rf, average="macro")

0.6253716946870129

In [47]:
# AUROC score for logistic regression
multiclass_roc_auc_score(y_test, y_pred_lr, average="macro")

0.5408427022290507

In [48]:
# AUROC score for Neural Network
multiclass_roc_auc_score(y_test, y_pred_nn, average="macro")

0.5

In [49]:
# AUROC score for K-Nearest neighbor
multiclass_roc_auc_score(y_test, y_pred_knn, average="macro")

0.5250579775327957

In [50]:
# AUROC score for Bayesian Network
multiclass_roc_auc_score(y_test, y_pred_bn, average="macro")

0.5267898300196612